# DEXTRA · 01b — CPCB station data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/01b_download_cpcb.ipynb)

Downloads the ground-truth pollutant readings — the only data in this project that is measured rather than modelled.

> ### ⚠️ This does not run on Colab
> CPCB drops connections from the cloud IP ranges Colab runs on. Every request times out at connect after 120 seconds, while the same URL answers in under a second from an ordinary Indian connection — a firewall, not a fault in this notebook.
>
> **Use `python scripts/fetch_cpcb.py --build --upload` on your own machine instead.** It runs this same code against the same manifest and sends the daily table to Hugging Face, which is all notebook 02 needs.
>
> This notebook is kept because the code is correct and the block is not ours: it works unchanged anywhere CPCB is reachable, including a local Jupyter kernel.

### What it fetches
`meta/cpcb_manifest.csv` holds the 76 stations this project uses — up to 6 per city, chosen for coverage first and then spaced apart, because a city value is the median across its stations and six sensors in one neighbourhood measure that neighbourhood rather than the city. `meta/cpcb_stations.tsv` holds the full 180-station master those were chosen from, with coordinates.

The run is resumable: anything already on Hugging Face is skipped.

In [ ]:
# Leave empty to read from the HF_DATASET_REPO Colab secret.
HF_DATASET_REPO = ""
GITHUB_REPO_URL = "https://github.com/Aduitya9/DEXTRA.git"

PAUSE_SECONDS = 0.5   # courtesy gap between requests to a public government server

## Secrets
The same two as notebooks 01 and 02: `HF_TOKEN` and `HF_DATASET_REPO` (🔑 in the left sidebar, **Notebook access** on).

In [ ]:
from google.colab import userdata

def colab_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret '{name}' is missing or this notebook can't read it. Open the key icon "
            "(Secrets) in the left sidebar, add it, and switch on 'Notebook access'."
        )
    return value

HF_TOKEN = colab_secret("HF_TOKEN")
HF_DATASET_REPO = HF_DATASET_REPO or colab_secret("HF_DATASET_REPO")
print("Dataset repo:", HF_DATASET_REPO)

In [ ]:
%pip install -q "huggingface_hub>=0.23.0" pyarrow

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/content/DEXTRA"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", GITHUB_REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import pandas as pd
from src.data import download_cpcb
from src.utils.hf_store import HFStore

store = HFStore(HF_DATASET_REPO, token=HF_TOKEN)
manifest = pd.read_csv(f"{REPO_DIR}/meta/cpcb_manifest.csv")
jobs = download_cpcb.manifest_jobs(manifest)
print(f"Connected to {HF_DATASET_REPO}")
print(f"{len(manifest)} stations across {manifest.city.nunique()} cities, {len(jobs)} station-years")
display(manifest.groupby("city").agg(stations=("site_id", "size"),
                                     files=("real_years", "sum"),
                                     spread_km=("km_from_centre", "max")))

## A · Download
One request per station-year, straight to Hugging Face. Already-uploaded files are skipped, so re-running after a dropped session costs only the listing.

Files go up a batch at a time, one commit per batch, because 478 separate commits would be slow and is the kind of rapid-fire write the Hub rate-limits.

A year the manifest expected to hold readings can still come back as the repository's all-NA template; those are reported rather than uploaded.

In [ ]:
summary = download_cpcb.download_manifest(manifest, store, pause=PAUSE_SECONDS)
print()
print(f"uploaded {summary['uploaded']}  skipped {summary['skipped']}  "
      f"empty {summary['empty']}  failed {summary['failed']}")

## B · What landed
Counts the files per city and reads one back, so a silent upload failure cannot pass for success.

In [ ]:
store.refresh()
landed = store.files("raw/cpcb/")
counts = pd.Series([p.split("/")[2] for p in landed]).value_counts().rename("files")
expected = manifest.groupby("city")["real_years"].sum()
expected.index = [c.lower() for c in expected.index]
check = pd.concat([counts, expected.rename("expected")], axis=1).fillna(0).astype(int)
check["ok"] = ["\u2713" if a >= b else "\u2717" for a, b in zip(check.files, check.expected)]
display(check.sort_index())
print(f"{len(landed)} files on Hugging Face")
if len(landed):
    # Reading one back proves the bytes survived the round trip. It must not be
    # able to end an hour of downloading in a traceback, so a failure is reported.
    try:
        from huggingface_hub import hf_hub_download
        from src.data.cpcb_loader import load_station_file
        local = hf_hub_download(HF_DATASET_REPO, landed[0], repo_type="dataset", token=HF_TOKEN)
        sample = load_station_file(local)
        print(f"\nread back {landed[0].split('/')[-1]}: {len(sample):,} hourly rows, "
              f"{sample['PM2.5'].notna().mean()*100:.0f}% of PM2.5 present")
    except Exception as exc:
        print(f"\n! could not read {landed[0]} back: {type(exc).__name__}: {exc}")
print("\nNext: run notebook 02 — section C will now find raw/cpcb/ and build the dataset.")